# 07 — ASRS LightGBM Signal Filter

## Objectif
Remplacer le filtre XGBoost (notebook 04) par un **LightGBM** avec early stopping.
Memes pipeline, memes features, memes splits — comparaison directe XGBoost vs LightGBM.

### Ameliorations vs notebook 04
- Early stopping (patience=50, `n_estimators=2000`) — evite l'overfitting
- `class_weight='balanced'` remplace `scale_pos_weight` manuel
- Analyse SHAP des features (valeurs de Shapley)
- Cellule de comparaison directe LightGBM vs XGBoost

> Baseline ASRS non-filtre (F1+F2+C4) : **WR 28.2%, PF 1.08**, +6 029 pts

In [4]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier, early_stopping, log_evaluation
except ImportError:
    raise ImportError('lightgbm non installe — uv add lightgbm')
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('shap non installe — uv add shap')
import joblib
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print(f'ROOT = {ROOT}')
print(f'LightGBM version : {lgb.__version__}')

ImportError: lightgbm non installe — uv add lightgbm

In [ ]:
# ── Trades ASRS ──────────────────────────────────────────────────────────────
trades_path = ROOT / 'data' / 'asrs_trades_all.csv'
assert trades_path.exists(), (
    f'Fichier trades introuvable : {trades_path}\n'
    'Executer le notebook 01_asrs_baseline.ipynb pour le generer.'
)
trades_raw = pd.read_csv(trades_path, parse_dates=['trade_date'])
trades_df  = trades_raw[trades_raw['variant'] == 'ASRS_4th'].copy()
trades_df  = trades_df.sort_values('trade_date').reset_index(drop=True)
print(f'Total trades (ASRS_4th) : {len(trades_df):,}')
print(f'Periode : {trades_df["trade_date"].min().date()} -> {trades_df["trade_date"].max().date()}')

# ── Donnees 5-min brutes ──────────────────────────────────────────────────────
raw_path = ROOT / 'data' / 'dax-5m_bk.csv'
assert raw_path.exists(), f'Donnees brutes introuvables : {raw_path}'
raw_df = pd.read_csv(
    raw_path, sep=';', header=None,
    names=['date', 'time', 'open', 'high', 'low', 'close', 'volume'],
    dtype={'date': str, 'time': str, 'open': float, 'high': float,
           'low': float, 'close': float, 'volume': float}
)
raw_df['dt'] = pd.to_datetime(raw_df['date'] + ' ' + raw_df['time'], format='%d/%m/%Y %H:%M')
raw_df = raw_df.sort_values('dt').reset_index(drop=True)
raw_df['date_only'] = raw_df['dt'].dt.date
print(f'Lignes 5-min : {len(raw_df):,}')

In [ ]:
import yfinance as yf

EXT_CACHE = ROOT / 'data' / 'ext_features.csv'

def download_ext_data(start='2005-01-01', end='2026-12-31'):
    tickers = {'^VIX': 'vix', 'EURUSD=X': 'eurusd', '^GSPC': 'spx'}
    frames = {}
    for ticker, name in tickers.items():
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        close = df['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        close = close.squeeze()
        close.name = name
        close.index = pd.to_datetime(close.index).tz_localize(None).normalize()
        frames[name] = close
    return pd.concat(frames.values(), axis=1).sort_index()

if EXT_CACHE.exists():
    ext_df = pd.read_csv(EXT_CACHE, index_col=0, parse_dates=True)
    print(f'Cache charge : {len(ext_df):,} lignes')
else:
    print('Telechargement VIX, EUR/USD, S&P 500 ...')
    ext_df = download_ext_data()
    ext_df.to_csv(EXT_CACHE)
    print(f'Cache sauvegarde : {len(ext_df):,} lignes')

ext_df['vix_prev']        = ext_df['vix'].shift(1)
ext_df['vix_5d_avg']      = ext_df['vix'].rolling(5).mean().shift(1)
ext_df['vix_5d_change']   = ext_df['vix'].shift(1) - ext_df['vix'].shift(6)
ext_df['spx_prev_ret']    = ext_df['spx'].pct_change().shift(1) * 100
ext_df['eurusd_prev_ret'] = ext_df['eurusd'].pct_change().shift(1) * 100
ext_df['eurusd_5d_ret']   = ext_df['eurusd'].pct_change(5).shift(1) * 100
ext_df.dropna(inplace=True)
print(f'Features externes : {list(ext_df.columns)}')

In [ ]:
# Identique notebook 04 — aucune modification
def _true_range(df):
    prev_close = df['close'].shift(1)
    return pd.concat([
        df['high'] - df['low'],
        (df['high'] - prev_close).abs(),
        (df['low']  - prev_close).abs(),
    ], axis=1).max(axis=1)

def _is_expiry_week(dt):
    import calendar
    third_fri = max(week[4] for week in calendar.monthcalendar(dt.year, dt.month) if week[4] != 0)
    return abs((dt - pd.Timestamp(dt.year, dt.month, third_fri)).days) <= 4

def build_features(trades_df, raw_df, ext_df):
    raw_df = raw_df.copy()
    raw_df['day'] = pd.to_datetime(raw_df['date_only'])
    grouped = {day: grp.reset_index(drop=True) for day, grp in raw_df.groupby('day')}
    daily = raw_df.groupby('day').agg(
        day_high=('high','max'), day_low=('low','min'),
        day_close=('close','last'), day_open=('open','first'), day_vol=('volume','sum')
    ).sort_index()
    daily['prev_close']     = daily['day_close'].shift(1)
    daily['prev_range']     = (daily['day_high'] - daily['day_low']).shift(1)
    daily['prev_close_pos'] = ((daily['day_close'] - daily['day_low']) /
                                (daily['day_high'] - daily['day_low'] + 1e-9)).shift(1)
    daily['prev_day_ret']   = daily['day_close'].pct_change().shift(1) * 100
    morn_vol = raw_df[
        (raw_df['dt'].dt.hour * 60 + raw_df['dt'].dt.minute >= 420) &
        (raw_df['dt'].dt.hour * 60 + raw_df['dt'].dt.minute <= 560)
    ].groupby('day')['volume'].sum().rolling(20, min_periods=5).mean().shift(1)
    rows = []
    for _, trade in trades_df.iterrows():
        td  = pd.Timestamp(trade['trade_date']).normalize()
        row = {}
        row['sig_range']      = trade['sig_range']
        row['bar_used']       = trade.get('bar_used', 0) or 0
        row['direction_enc']  = 1 if str(trade['direction']).lower() in ('long','buy') else 0
        row['dow']            = td.dayofweek
        row['month']          = td.month
        row['is_expiry_week'] = int(_is_expiry_week(td))
        row['week_of_month']  = (td.day - 1) // 7 + 1
        bdays = pd.bdate_range(td.replace(day=1), td + pd.offsets.MonthEnd(0))
        row['is_month_end']   = int(td in bdays and bdays.get_loc(td) >= len(bdays) - 3)
        row['is_month_start'] = int(td in bdays and bdays.get_loc(td) <= 2)
        ext_row = ext_df[ext_df.index <= td].tail(1)
        if len(ext_row):
            er = ext_row.iloc[0]
            row['vix_prev']        = er['vix_prev']
            row['vix_regime']      = 0 if er['vix_prev'] < 15 else 1 if er['vix_prev'] < 25 else 2 if er['vix_prev'] < 35 else 3
            row['vix_5d_change']   = er['vix_5d_change']
            row['spx_prev_ret']    = er['spx_prev_ret']
            row['eurusd_prev_ret'] = er['eurusd_prev_ret']
            row['eurusd_5d_ret']   = er['eurusd_5d_ret']
        else:
            for k in ['vix_prev','vix_regime','vix_5d_change','spx_prev_ret','eurusd_prev_ret','eurusd_5d_ret']:
                row[k] = np.nan
        day_bars = grouped.get(td)
        if day_bars is None or len(day_bars) == 0:
            for k in ['sig_range_norm','bar_body_ratio','wick_upper_ratio','wick_lower_ratio',
                      'bar_bullish','atr14_abs','atr5_abs','vol_ratio','prev_day_range',
                      'prev_close_pos','prev_day_ret','gap_pts','overnight_ret',
                      'morning_range','morning_direction','morning_vol_ratio','price_vs_ma20','consecutive_up']:
                row[k] = np.nan
            row['winner'] = 1 if trade['pnl'] > 0 else 0
            rows.append(row); continue
        sig_mask = day_bars['dt'].dt.hour * 60 + day_bars['dt'].dt.minute == 555
        sig_bars = day_bars[sig_mask]
        sig_bar  = sig_bars.iloc[0] if len(sig_bars) else None
        sig_idx  = sig_bars.index[0] if len(sig_bars) else None
        pre_bars = day_bars.loc[:sig_idx] if sig_idx is not None else day_bars
        tr    = _true_range(pre_bars)
        atr14 = tr.rolling(14, min_periods=1).mean().iloc[-1]
        atr5  = tr.rolling(5,  min_periods=1).mean().iloc[-1]
        ma20  = pre_bars['close'].rolling(20, min_periods=1).mean().iloc[-1]
        row['atr14_abs']      = atr14
        row['atr5_abs']       = atr5
        row['vol_ratio']      = atr5 / atr14 if atr14 > 0 else 1.0
        row['sig_range_norm'] = trade['sig_range'] / atr14 if atr14 > 0 else np.nan
        if sig_bar is not None:
            sig_o,sig_h,sig_l,sig_c = sig_bar['open'],sig_bar['high'],sig_bar['low'],sig_bar['close']
            r = sig_h - sig_l; d = r if r > 0 else 1.0
            row['bar_body_ratio']   = abs(sig_c - sig_o) / d
            row['wick_upper_ratio'] = (sig_h - max(sig_o, sig_c)) / d
            row['wick_lower_ratio'] = (min(sig_o, sig_c) - sig_l) / d
            row['bar_bullish']      = 1 if sig_c > sig_o else 0
            row['price_vs_ma20']    = (sig_c - ma20) / atr14 if atr14 > 0 else 0.0
        else:
            for k in ['bar_body_ratio','wick_upper_ratio','wick_lower_ratio','bar_bullish','price_vs_ma20']:
                row[k] = np.nan
        if td in daily.index:
            d_row = daily.loc[td]
            row['prev_day_range'] = d_row['prev_range']
            row['prev_close_pos'] = d_row['prev_close_pos']
            row['prev_day_ret']   = d_row['prev_day_ret']
            prev_close            = d_row['prev_close']
        else:
            row['prev_day_range'] = row['prev_close_pos'] = row['prev_day_ret'] = np.nan
            prev_close = np.nan
        open_900 = day_bars[day_bars['dt'].dt.hour * 60 + day_bars['dt'].dt.minute == 540]
        if len(open_900) and not np.isnan(prev_close):
            row['gap_pts']       = open_900.iloc[0]['open'] - prev_close
            row['overnight_ret'] = (open_900.iloc[0]['open'] / prev_close - 1) * 100 if prev_close else np.nan
        else:
            row['gap_pts'] = row['overnight_ret'] = np.nan
        morn = day_bars[
            (day_bars['dt'].dt.hour * 60 + day_bars['dt'].dt.minute >= 540) &
            (day_bars['dt'].dt.hour * 60 + day_bars['dt'].dt.minute <= 555)
        ]
        row['morning_range']     = morn['high'].max() - morn['low'].min() if len(morn) >= 1 else np.nan
        row['morning_direction'] = float(np.sign(morn.iloc[-1]['close'] - morn.iloc[0]['open'])) if len(morn) >= 1 else np.nan
        tmv = day_bars[
            (day_bars['dt'].dt.hour * 60 + day_bars['dt'].dt.minute >= 420) &
            (day_bars['dt'].dt.hour * 60 + day_bars['dt'].dt.minute <= 560)
        ]['volume'].sum()
        avg_morn = morn_vol.get(td, np.nan)
        row['morning_vol_ratio'] = tmv / avg_morn if avg_morn and avg_morn > 0 else np.nan
        past_days = sorted([d for d in daily.index if d < td])[-5:]
        consec = 0
        if len(past_days) >= 2:
            rets = [daily.loc[d,'prev_day_ret'] for d in past_days if d in daily.index]
            last_sign = np.sign(rets[-1]) if rets else 0
            for rv in reversed(rets):
                if np.sign(rv) == last_sign: consec += 1
                else: break
        row['consecutive_up'] = consec * (1 if len(past_days) > 0 and daily.loc[past_days[-1],'prev_day_ret'] > 0 else -1) if consec > 0 else 0
        row['winner'] = 1 if trade['pnl'] > 0 else 0
        rows.append(row)
    feat_df = pd.DataFrame(rows).reset_index(drop=True).ffill().fillna(0)
    feat_df['trade_date']   = trades_df['trade_date'].values
    feat_df['pnl']          = trades_df['pnl'].values
    feat_df['initial_risk'] = trades_df['initial_risk'].values
    return feat_df

print('build_features() definie — 30+ features.')

In [ ]:
print('Construction matrice de features (2-3 min) ...')
features_df = build_features(trades_df, raw_df, ext_df)
print(f'Shape : {features_df.shape}')

FEATURE_COLS = [
    'sig_range','sig_range_norm','bar_body_ratio','wick_upper_ratio','wick_lower_ratio','bar_bullish','bar_used',
    'atr14_abs','atr5_abs','vol_ratio',
    'prev_day_range','prev_close_pos','prev_day_ret','gap_pts','overnight_ret',
    'morning_range','morning_direction','morning_vol_ratio','price_vs_ma20',
    'vix_prev','vix_regime','vix_5d_change','spx_prev_ret','eurusd_prev_ret','eurusd_5d_ret',
    'dow','month','week_of_month','is_expiry_week','is_month_end','is_month_start',
    'direction_enc','consecutive_up',
]

n_win  = (features_df['winner'] == 1).sum()
n_loss = (features_df['winner'] == 0).sum()
print(f'Repartition : Gagnants {n_win:,} ({100*n_win/len(features_df):.1f}%) | Perdants {n_loss:,} ({100*n_loss/len(features_df):.1f}%)')
print('\nTop 15 correlations avec winner :')
corr = features_df[FEATURE_COLS + ['winner']].corr()['winner'].drop('winner').sort_values(key=abs, ascending=False)
for feat, val in corr.head(15).items():
    print(f'  {feat:<25} {val:+.4f}')

In [ ]:
features_df['vol_regime'] = pd.qcut(features_df['atr14_abs'], q=3, labels=['low','mid','high'])
print(f'  {"Regime":<8}  {"N":>9}  {"WR":>8}  {"ATR14 moy":>10}')
print('-' * 42)
for regime in ['low','mid','high']:
    mask = features_df['vol_regime'] == regime
    print(f'  {regime:<8}  {mask.sum():>9,}  {features_df.loc[mask,"winner"].mean()*100:>7.1f}%  {features_df.loc[mask,"atr14_abs"].mean():>10.1f}')

In [ ]:
FEATURES = FEATURE_COLS
TARGET   = 'winner'
X    = features_df[FEATURES].values
y    = features_df[TARGET].values
X_df = features_df[FEATURES]
strat_labels = features_df['vol_regime'].astype(str) + '_' + features_df[TARGET].astype(str)

LGB_PARAMS = dict(
    n_estimators      = 2000,
    learning_rate     = 0.02,
    num_leaves        = 31,
    max_depth         = -1,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    class_weight      = 'balanced',
    random_state      = 42,
    verbose           = -1,
    n_jobs            = -1,
)

cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs  = np.zeros(len(X))
fold_aucs  = []
best_iters = []

print('LightGBM — 5-fold CV avec early stopping (patience=50) ...')
for fold, (train_idx, val_idx) in enumerate(cv.split(X, strat_labels)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    clf = LGBMClassifier(**LGB_PARAMS)
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(50, verbose=False), log_evaluation(0)],
    )
    oof_probs[val_idx] = clf.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, oof_probs[val_idx])
    fold_aucs.append(auc)
    best_iters.append(clf.best_iteration_)
    print(f'  Fold {fold+1}: AUC={auc:.4f}  best_iter={clf.best_iteration_}  (N={len(val_idx)}, WR={y_val.mean()*100:.1f}%)')

overall_auc    = roc_auc_score(y, oof_probs)
mean_best_iter = int(np.mean(best_iters))
print(f'\nAUC OOF global   : {overall_auc:.4f}')
print(f'Mean AUC +/- std : {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}')
print(f'Iterations moy.  : {mean_best_iter}')

In [ ]:
thresholds = np.arange(0.30, 0.70, 0.05)
all_pnl    = features_df['pnl'].values
all_wins   = features_df['winner'].values
pf_base    = all_pnl[all_pnl > 0].sum() / abs(all_pnl[all_pnl < 0].sum())
print(f'Baseline : N={len(all_pnl):,}  WR={all_wins.mean()*100:.1f}%  PF={pf_base:.3f}  PnL={all_pnl.sum():+.0f}\n')
print(f'{"Seuil":>7} {"N garde":>8} {"Filtre%":>9} {"WR%":>7} {"PF":>7} {"PnL":>11}')
print('-' * 57)
results = []
for thr in thresholds:
    keep = oof_probs >= thr
    n_k  = keep.sum()
    if n_k == 0: continue
    pnl_k = all_pnl[keep]
    wr    = all_wins[keep].mean() * 100
    gw    = pnl_k[pnl_k > 0].sum()
    gl    = abs(pnl_k[pnl_k < 0].sum())
    pf    = gw / gl if gl > 0 else float('inf')
    tot   = pnl_k.sum()
    print(f'{thr:>7.2f} {n_k:>8,} {(1-n_k/len(all_pnl))*100:>8.1f}% {wr:>6.1f}% {pf:>7.3f} {tot:>+11.0f}')
    results.append({'threshold': thr, 'n_kept': n_k, 'wr': wr, 'pf': pf, 'total_pnl': tot})
results_df = pd.DataFrame(results)
valid      = results_df[results_df['n_kept'] >= 200]
best_row   = valid.loc[valid['pf'].idxmax()] if len(valid) > 0 else results_df.iloc[-1]
BEST_THRESHOLD = best_row['threshold']
print(f'\nMeilleur seuil (PF-max, N>=200) : {BEST_THRESHOLD:.2f}')
print(f'  N={int(best_row["n_kept"]):,}  WR={best_row["wr"]:.1f}%  PF={best_row["pf"]:.3f}  PnL={best_row["total_pnl"]:+.0f}')

In [ ]:
filter_mask  = oof_probs >= BEST_THRESHOLD
dates_all    = pd.to_datetime(features_df['trade_date'])
cum_all      = np.cumsum(all_pnl)
cum_filtered = np.cumsum(all_pnl[filter_mask])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(dates_all,              cum_all,      lw=1.2, color='steelblue',  label='ASRS non filtre')
ax.plot(dates_all[filter_mask], cum_filtered, lw=1.6, color='darkorange', label=f'ASRS + LightGBM (seuil={BEST_THRESHOLD:.2f})')
ax.axhline(0, color='grey', lw=0.6)
ax.set_title('ASRS — Equity curve : non filtre vs LightGBM (OOF)')
ax.set_xlabel('Date'); ax.set_ylabel('PnL cumule (pts)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Non filtre : N={len(all_pnl):,}  PnL={cum_all[-1]:+.0f}')
print(f'LightGBM   : N={filter_mask.sum():,}  PnL={cum_filtered[-1] if len(cum_filtered)>0 else 0:+.0f}')

In [ ]:
print('Entrainement du modele final ...')
final_model = LGBMClassifier(**{**LGB_PARAMS, 'n_estimators': mean_best_iter})
final_model.fit(X, y)

feat_imp_df = pd.DataFrame({'feature': FEATURES, 'importance': final_model.feature_importances_})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_imp_df['feature'][::-1], feat_imp_df['importance'][::-1], color='seagreen')
ax.set_xlabel('Feature importance (gain)')
ax.set_title('Top 15 features — ASRS LightGBM')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

print(f'{"Rang":<5} {"Feature":<25} {"Importance":>12}')
print('-' * 44)
for rank, (_, row) in enumerate(feat_imp_df.iterrows(), 1):
    print(f'{rank:<5} {row["feature"]:<25} {row["importance"]:>12.1f}')

In [ ]:
if not SHAP_AVAILABLE:
    print('shap non disponible — uv add shap')
else:
    print('Calcul valeurs SHAP ...')
    explainer   = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_df)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    mean_abs_shap = np.abs(sv).mean(axis=0)
    shap_imp = pd.DataFrame({'feature': FEATURES, 'mean_abs_shap': mean_abs_shap})
    shap_imp = shap_imp.sort_values('mean_abs_shap', ascending=False).head(15)
    top15     = shap_imp['feature'].tolist()
    top15_idx = [FEATURES.index(f) for f in top15]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].barh(shap_imp['feature'][::-1], shap_imp['mean_abs_shap'][::-1], color='salmon')
    axes[0].set_xlabel('Impact moyen |SHAP|')
    axes[0].set_title('Importance SHAP (absolue moyenne)')
    axes[0].grid(True, axis='x', alpha=0.3)
    plt.sca(axes[1])
    shap.summary_plot(sv[:, top15_idx], X_df[top15], plot_type='dot', show=False, plot_size=None)
    axes[1].set_title('SHAP beeswarm — top 15')
    plt.tight_layout(); plt.show()

    print(f'{"Rang":<5} {"Feature":<25} {"SHAP moyen":>12}')
    print('-' * 44)
    for rank, (_, row) in enumerate(shap_imp.iterrows(), 1):
        print(f'{rank:<5} {row["feature"]:<25} {row["mean_abs_shap"]:>12.5f}')

In [ ]:
def regime_stats(mask, label):
    pnl  = features_df.loc[mask, 'pnl'].values
    wins = features_df.loc[mask, 'winner'].values
    if len(pnl) == 0: return
    gw = pnl[pnl > 0].sum(); gl = abs(pnl[pnl < 0].sum())
    pf = gw / gl if gl > 0 else float('inf')
    print(f'{label:<24} {len(pnl):>8,}  {wins.mean()*100:>7.1f}%  {pf:>7.3f}  {pnl.sum():>+12.0f}')

print(f'{"Label":<24} {"N":>8}  {"WR":>7}  {"PF":>7}  {"PnL":>12}')
print('-' * 66)
for regime in ['low','mid','high']:
    rm = features_df['vol_regime'] == regime
    regime_stats(rm,               f'{regime.upper()} VOL non filtre')
    regime_stats(rm & filter_mask,  f'{regime.upper()} VOL LightGBM')
    print()

In [ ]:
TRAIN_END  = pd.Timestamp('2019-12-31')
TEST_START = pd.Timestamp('2020-01-01')
dates      = pd.to_datetime(features_df['trade_date'])
train_mask = dates <= TRAIN_END
test_mask  = dates >= TEST_START
X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]
print(f'Train 2006-2019 : {train_mask.sum():,} trades')
print(f'Test  2020-2026 : {test_mask.sum():,} trades')

n_val    = int(len(X_train) * 0.15)
wf_model = LGBMClassifier(**LGB_PARAMS)
wf_model.fit(
    X_train[:-n_val], y_train[:-n_val],
    eval_set=[(X_train[-n_val:], y_train[-n_val:])],
    callbacks=[early_stopping(50, verbose=False), log_evaluation(0)],
)
print(f'Walk-forward best_iter = {wf_model.best_iteration_}')

wf_probs  = wf_model.predict_proba(X_test)[:, 1]
wf_filter = wf_probs >= BEST_THRESHOLD
pnl_test  = features_df.loc[test_mask, 'pnl'].values
wins_test = features_df.loc[test_mask, 'winner'].values
pnl_filt  = pnl_test[wf_filter]
wins_filt = wins_test[wf_filter]

def pf(arr): return arr[arr>0].sum() / abs(arr[arr<0].sum()) if abs(arr[arr<0].sum()) > 0 else float('inf')

test_auc = roc_auc_score(y_test, wf_probs)
print(f'AUC test (walk-forward) : {test_auc:.4f}\n')
print(f'{"":22} {"N":>7} {"WR%":>7} {"PF":>7} {"PnL":>11}')
print('-' * 56)
print(f'{"ASRS non filtre":<22} {len(pnl_test):>7,} {wins_test.mean()*100:>6.1f}% {pf(pnl_test):>7.3f} {pnl_test.sum():>+11.0f}')
if len(pnl_filt) > 0:
    print(f'{"ASRS + LightGBM":<22} {len(pnl_filt):>7,} {wins_filt.mean()*100:>6.1f}% {pf(pnl_filt):>7.3f} {pnl_filt.sum():>+11.0f}')
else:
    print('Aucun trade passe le filtre sur la periode de test.')

In [ ]:
xgb_model_path = ROOT / 'models' / 'asrs_xgb_filter.pkl'
xgb_feats_path = ROOT / 'models' / 'asrs_xgb_features.pkl'

if xgb_model_path.exists() and xgb_feats_path.exists():
    xgb_model     = joblib.load(xgb_model_path)
    xgb_feats     = joblib.load(xgb_feats_path)
    XGB_THRESHOLD = 0.65
    xgb_probs     = xgb_model.predict_proba(features_df[xgb_feats].values)[:, 1]
    xgb_filter    = xgb_probs >= XGB_THRESHOLD
    lgb_filter    = oof_probs >= BEST_THRESHOLD

    def stats(mask, label):
        pnl_m = all_pnl[mask]; wins_m = all_wins[mask]
        if len(pnl_m) == 0: return
        gw = pnl_m[pnl_m > 0].sum(); gl = abs(pnl_m[pnl_m < 0].sum())
        pf_v = gw / gl if gl > 0 else float('inf')
        print(f'{label:<32} {len(pnl_m):>7,}  {wins_m.mean()*100:>6.1f}%  {pf_v:>7.3f}  {pnl_m.sum():>+11.0f}')

    print(f'{"Modele":<32} {"N":>7}  {"WR%":>6}  {"PF":>7}  {"PnL":>11}')
    print('-' * 67)
    stats(np.ones(len(all_pnl), dtype=bool), 'ASRS non filtre (baseline)')
    stats(xgb_filter,               f'XGBoost  (seuil={XGB_THRESHOLD:.2f})')
    stats(lgb_filter,               f'LightGBM (seuil={BEST_THRESHOLD:.2f})')
    stats(xgb_filter & lgb_filter,  'XGBoost ET LightGBM (consensus)')

    xgb_auc = roc_auc_score(y, xgb_probs)
    print(f'\nAUC OOF LightGBM : {overall_auc:.4f}')
    print(f'AUC full XGBoost : {xgb_auc:.4f}  (indicatif)')

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(dates_all,              np.cumsum(all_pnl),              lw=1.2, color='steelblue',  label='Non filtre')
    ax.plot(dates_all[xgb_filter],  np.cumsum(all_pnl[xgb_filter]), lw=1.4, color='darkorange', label=f'XGBoost (thr={XGB_THRESHOLD:.2f})')
    ax.plot(dates_all[lgb_filter],  np.cumsum(all_pnl[lgb_filter]), lw=1.4, color='seagreen',   label=f'LightGBM (thr={BEST_THRESHOLD:.2f})', linestyle='--')
    ax.axhline(0, color='grey', lw=0.6)
    ax.set_title('ASRS — Comparaison XGBoost vs LightGBM (OOF)')
    ax.set_xlabel('Date'); ax.set_ylabel('PnL cumule (pts)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("Modele XGBoost non trouve — executer le notebook 04 d'abord.")

In [ ]:
models_dir = ROOT / 'models'
models_dir.mkdir(exist_ok=True)
joblib.dump(final_model, models_dir / 'asrs_lgb_filter.pkl')
joblib.dump(FEATURES,    models_dir / 'asrs_lgb_features.pkl')
print('Modele sauvegarde  -> models/asrs_lgb_filter.pkl')
print('Features           -> models/asrs_lgb_features.pkl')
print(f'Seuil optimal      : {BEST_THRESHOLD:.2f}')
print(f'AUC OOF            : {overall_auc:.4f}')
print(f'Iterations         : {mean_best_iter}')
print()
print('Usage production :')
print("  model    = joblib.load('models/asrs_lgb_filter.pkl')")
print("  features = joblib.load('models/asrs_lgb_features.pkl')")
print("  prob     = model.predict_proba(X_new)[0, 1]")
print(f"  trade    = prob >= {BEST_THRESHOLD:.2f}")